# IND320 project 2026
Philip A. K. Øvland \
Github:     https://github.com/Povland/IND320 \
Streamlit:  https://ind320-2026.streamlit.app/

Video explaining stuff (Please tell me if the link doesn't work): TO DO

## AI Usage
Claude (Anthropic) was used as a coding assistant: It helped me with planning, checking API costs, drafting code file by file, reviewing the repository and drafting notebook text. I decided the structure and features, ran and tested everything myself, and adjusted code and text where needed.

## Log - Tasks
**Planning.** Part 4 started by reading the task, checking what part 3 left, and deciding the structure before coding.

**Data.** Weather was downloaded for all areas for 2024 and 2025, with **only 100 locations per area**, to stay within Open-Meteo's free limit (one more year costs about 6,700 of 10,000 daily calls). So the weather-based parts only cover those two years.

**Structure.** The package-first design from part 3 held up. New logic is in `src/ind320` (`norgesbank.py`, `forecast.py`, `correlation.py`, weekly weather, GeoJSON). The notebook tests and explains it, and the app adds caching and error handling in `app/utils/data.py`.

**Forecasting.** SNARIMAX from River is SARIMAX trained online: a linear regression on lagged values with SGD, so it is also the lag-based machine learning. Weekly resolution makes each run take milliseconds.

**The app** is grouped by type of task (Explore, Anomalies, Predict, Data). Map of Norway has the GeoJSON map, colouring by filling degree or weather, a week shared with the forecast, clicks that select an area for other pages, and cross-border flow arrows. Data status is the one place to check and fetch every source. We kept Streamlit's default theme, with the same blue and orange in all plots. The LineChartColumn table stayed as the Data tab on Reservoirs.

**Struggles.** The old animated map redrew every polygon in every frame and snapped on every change. Coarser outlines and a slider that runs in the browser fixed it. Smaller ones: a file saved as UTF-16, `nbformat` removed by `uv sync`, and kernel restarts until autoreload was added.

**Bonus.** Both *Waiting time* (spinners, progress bars, a busy cursor, caching of loads, figures, forecasts and correlations) and *Error handling* (checks of every source, readable messages, NaN handling, a safety net on every page). Confidence intervals are also in the forecast.

# Setup
Imports used throughout the notebook. The logic lives in the `ind320` package (`src/ind320/`), installed with `uv sync`. The notebook calls it step by step and shows the results, and the Streamlit app uses the same functions.

Plotly figures are shown with the `notebook` renderer, so they are embedded in the HTML export (VS Code's own plotly format is not).

In [ ]:
# Reloads changed modules from src/ind320 automatically before each cell, so edits are seen without a restart
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.io as pio

from ind320 import areas

# Embeds plotly figures as HTML/JavaScript, so they are kept when the notebook is exported to HTML
pio.renderers.default = "notebook"

DATA_PATH = areas.DATA_PATH

print("Python:", sys.version.split()[0])
print("Running from:", Path.cwd())
print("Data folder:", DATA_PATH)

# App structure and navigation
The sketch made before changing the code. Pages are ordered by type of task. Home and the map stand above the groups, since a click on the map selects the area for all of them.

```
Home                Intro, small map, links, data sources
Map of Norway       Main map: colour by filling or weather, week, click to select, flow arrows
EXPLORE             Reservoirs, Cross-border flow, Currency, Seasonality & Spectrum
ANOMALIES           Weather Outliers
PREDICT             Weather vs reservoirs, Forecast
DATA                Data status, About
```

- **Sidebar** (area type, areas, period): shared by Reservoirs, Seasonality, Weather Outliers and Weather vs reservoirs.
- **Click on the map:** sets the selected area for every page.
- **Week on the map = last training week on Forecast** (`session_state["selected_week"]`).
- **Local choices** (weather property, model settings) are remembered per page.

# API: Norges Bank (currency)
EUR and USD in NOK from Norges Bank's open API (`ind320.norgesbank`), daily, and weekly with weeks ending on Sunday like NVE's.

In [ ]:
# Exchange rates from Norges Bank: daily, then weekly (weeks ending on Sunday, as NVE's)
from ind320 import norgesbank

daily = norgesbank.fetch_rates("2024-01-01", "2024-03-31")
display(daily.head())
print(f"{len(daily)} business days, {daily.isna().sum().sum()} missing values")

weekly_rates = norgesbank.weekly_rates("2024-01-01", "2024-03-31")
display(weekly_rates.head())

# Areas as GeoJSON
The shapefiles are converted to one GeoJSON file per area type (`data/areas_*.geojson`), clipped to land and simplified to about 1 km. The app's maps read these.

In [ ]:
# Writes the three GeoJSON files to data/, and reads them back to check them
paths = areas.export_geojson()
for area_type, path in paths.items():
    shapes = areas.load_geojson(area_type)
    print(f"{area_type}: {len(shapes)} areas, {path.stat().st_size / 1e3:.0f} kB, {path.name}")

# Time zones
How each source delivers time, and what daylight saving does to a Norwegian day.

In [ ]:
from pymongo import MongoClient

from ind320 import norgesbank, nve, weather
from ind320.config import load_env

mongo_db = MongoClient(load_env("atlas-credentials.env")["MONGODB_URI"], serverSelectionTimeoutMS=10000)["ind320"]

# NVE: one date per week, no time and no time zone. Every date is a Sunday (the end of the ISO week)
reservoirs_tz = nve.fetch_reservoir_data()
print("NVE dates:", reservoirs_tz.index[-2:].tolist(), "| time zone:", reservoirs_tz.index.tz,
      "| weekdays:", reservoirs_tz.index.day_name().unique().tolist())
print("NVE next publication:", reservoirs_tz["next_publishing_date"].iloc[-1])

# Open-Meteo: requested in GMT (UTC), stored in MongoDB in UTC
hourly_2024 = weather.load_region_weather(mongo_db, "NO1", 2024)
print("Open-Meteo:", hourly_2024.index[:2].tolist())

# ENTSO-E: stored in MongoDB in UTC (MongoDB gives the times back without a time zone mark)
flow_document = mongo_db["crossborder_flows"].find_one({}, {"_id": 0, "connection": 1, "time": {"$slice": 2}})
print("ENTSO-E:", flow_document["connection"], flow_document["time"])

# Norges Bank: one date per business day, no time and no time zone
rates_tz = norgesbank.fetch_rates("2024-03-25", "2024-04-05")
print("Norges Bank:", rates_tz.index[:3].tolist(), "| time zone:", rates_tz.index.tz)

# Daylight saving time: in Norwegian time, 31 March 2024 has 23 hours and 27 October 2024 has 25
local_hours = hourly_2024.iloc[:, 0].tz_convert("Europe/Oslo").resample("D").count()
display(local_hours.loc[["2024-03-30", "2024-03-31", "2024-10-27"]])

| Source | Resolution | Time zone |
|---|---|---|
| NVE | weekly, Sunday dates | none (Norwegian weeks) |
| ENTSO-E | 15 min / hourly | stored in UTC |
| Open-Meteo | hourly | UTC |
| Norges Bank | business days, dates | none |

**Problems:** NVE's weeks are Norwegian (Monday to Sunday, local time), while the hourly data is UTC, so grouping in UTC puts Monday's first hours in the wrong week. A Norwegian day has 23 or 25 hours on the daylight saving days. Hourly `W-SUN` grouping puts Sunday's hours in the next week.

**Solutions** (`weather.weekly_weather`, `load_weekly_flows`): store in UTC, convert to Europe/Oslo before grouping, group hours into days and then weeks ending on Sunday, accept days with 23 or more hours, and drop the time zone so everything joins on NVE's dates.

# Forecasting the filling degree (SARIMAX with River)
`forecast.forecast_filling` forecasts the weekly filling degree of one area with SNARIMAX from River (online SARIMAX). Its inputs:

- **Model parameters:** `p`, `d`, `q` (AR order, differencing, MA order), `m`, `sp`, `sd`, `sq` (season length and the seasonal orders), `learning_rate`
- **Length of history:** `history_weeks`, counted back from `train_end`
- **Horizon:** `horizon`, the number of weeks after `train_end`
- **Data sources:** `exog`, a table with one column per exogenous variable (weekly weather, or e.g. `norgesbank.weekly_rates`), and `exog_lag`. No `exog` means the filling degree only

The forecast is dynamic: each forecast week is fed back as input for the next. With `intervals=True`, two extra models trained on the quantile loss give a confidence interval (the bonus task). It returns the weeks with observed, fitted and forecast values, and the errors against what happened and against the baseline.

In [ ]:
from ind320 import forecast, nve, seasonal

# The weekly filling degree of NO1, straight from NVE's API
reservoirs = nve.fetch_reservoir_data()
filling = seasonal.filling_series(reservoirs, "EL", 1)

# Without exogenous variables: train on 10 years up to 30 June 2025, forecast 26 weeks (a backtest,
# so the forecast can be compared with what happened)
result, info = forecast.forecast_filling(filling, train_end="2025-06-29", history_weeks=520, horizon=26,
                                         intervals=True)
forecast.forecast_figure(result, info, title="NO1: forecast without exogenous variables").show()
{key: info[key] for key in ["train_weeks", "mae_fit", "mae_forecast", "mae_naive", "seconds"]}

In [ ]:
from pymongo import MongoClient

from ind320 import weather
from ind320.config import load_env

# NO1 has weather for both 2024 and 2025 (100 locations), the same area as in the cell above
REGION, AREA_TYPE, AREA_NR = "NO1", "EL", 1
creds = load_env("atlas-credentials.env")
mongo_db = MongoClient(creds["MONGODB_URI"], serverSelectionTimeoutMS=10000)["ind320"]

# load_region_weather gives None for a year that isn't stored, and pd.concat would skip it without a word
yearly = {year: weather.load_region_weather(mongo_db, REGION, year) for year in [2024, 2025]}
missing_years = [year for year, data in yearly.items() if data is None]
assert not missing_years, f"{REGION}: no weather stored for {missing_years}"

weekly = weather.weekly_weather(pd.concat(yearly.values()))
print(f"{len(weekly)} complete weeks, {weekly.index[0]:%d.%m.%Y} to {weekly.index[-1]:%d.%m.%Y}")
display(weekly.head())

filling_area = seasonal.filling_series(reservoirs, AREA_TYPE, AREA_NR)
exog = weekly[["temperature_2m", "precipitation"]]

# Two settings with weather, since only 1.5 years of weather is available:
#   seasonal differencing (sd=1): the first 52 weeks only go to building "last year", so it learns from about half a year
#   no seasonal differencing (sd=0, d=1): learns from almost every week, and the weather has to carry the season
settings = {
    "seasonal differencing (sd=1)": dict(),
    "no seasonal differencing (sd=0, d=1)": dict(d=1, sd=0),
}
runs = {name: forecast.forecast_filling(filling_area, exog=exog, train_end="2025-06-29", horizon=26,
                                        intervals=True, **extra)
        for name, extra in settings.items()}

# The two side by side, then the plot of the one with the lowest forecast error
comparison = pd.DataFrame({name: info for name, (_, info) in runs.items()}).T
display(comparison[["train_weeks", "exog_source", "mae_fit", "mae_forecast", "mae_naive", "seconds"]])

best = comparison["mae_forecast"].astype(float).idxmin()
result, info = runs[best]
forecast.forecast_figure(result, info, title=f"{REGION}: with weather, {best}").show()

## Time resolution
Weekly, since NVE publishes weekly: a finer series would only be interpolated values. The weather is summed up per week (temperature as a mean, the rest as sums), and m = 52 gives one year back. A run takes milliseconds, so the app can forecast on every change.

Measured: one forecast takes about 0.03 to 0.05 seconds, and the whole first grid (60 backtests with up to 30 years) about 3.5 seconds [ ].

## Testing different settings
Each setting is tested as a **backtest**: train up to a date, forecast 26 weeks, and compare with what really happened. Several start points are used, and the error is averaged, so one unusual year doesn't decide the result.

- **MAE** (mean absolute error) is in filling degree: 0.05 means 5 percentage points off on average.
- **The baseline** is the seasonal naive forecast: the same week last year. A model that can't beat it is not worth using.
- **skill** = 1 − MAE / MAE of the baseline. Positive is better than the baseline, negative is worse.

Two grids:
1. **Without exogenous variables**, where the whole history since 1995 can be used: model settings and history length.
2. **With weather**, which is only available for 2024–2025: which weather variables help.

In [ ]:
import itertools

# Model settings to compare. All have m = 52 (one year of weeks)
MODELS = {
    "AR(2), seasonal diff.":           dict(p=2, d=0, q=0, sp=0, sd=1, sq=0),
    "AR(4), seasonal diff.":           dict(p=4, d=0, q=0, sp=0, sd=1, sq=0),
    "AR(2) + seasonal AR(1), s. diff.": dict(p=2, d=0, q=0, sp=1, sd=1, sq=0),
    "AR(2) + MA(1), seasonal diff.":   dict(p=2, d=0, q=1, sp=0, sd=1, sq=0),
    "AR(2), diff. (no seasonal diff.)": dict(p=2, d=1, q=0, sp=0, sd=0, sq=0),
}
HISTORIES = {"5 years": 260, "10 years": 520, "all (since 1995)": None}
ORIGINS = ["2023-06-25", "2024-06-30", "2025-06-29"]      # training ends here, then 26 weeks are forecast
LEARNING_RATES = [0.01, 0.05]

rows = []
for (model_name, model), (history_name, weeks), lr, origin in itertools.product(
        MODELS.items(), HISTORIES.items(), LEARNING_RATES, ORIGINS):
    try:
        _, info = forecast.forecast_filling(filling, train_end=origin, history_weeks=weeks, horizon=26,
                                            learning_rate=lr, **model)
    except ValueError as error:          # a combination that doesn't have enough data
        print(f"Skipped {model_name}, {history_name}, {origin}: {error}")
        continue
    rows.append({"model": model_name, "history": history_name, "learning_rate": lr, "origin": origin,
                 "mae_forecast": info["mae_forecast"], "mae_naive": info["mae_naive"], "seconds": info["seconds"]})

grid = pd.DataFrame(rows)

# Averaged over the start points, best first
summary = (grid.groupby(["model", "history", "learning_rate"])
               .agg(mae_forecast=("mae_forecast", "mean"), mae_naive=("mae_naive", "mean"), seconds=("seconds", "mean"))
               .assign(skill=lambda t: 1 - t["mae_forecast"] / t["mae_naive"])
               .sort_values("mae_forecast"))
print(f"{len(grid)} runs, {grid['seconds'].sum():.0f} seconds in total")
summary.round(4).head(10)

In [ ]:
# Which weather variables help. Only 2024-2025 is stored, so the training starts in January 2024.
# Two start points, and the two model settings that can train on such a short period
EXOG_SETS = {
    "none":                      [],
    "temperature":               ["temperature_2m"],
    "precipitation":             ["precipitation"],
    "temperature + precipitation": ["temperature_2m", "precipitation"],
    "temp. + precip. + snowfall": ["temperature_2m", "precipitation", "snowfall"],
}
SHORT_MODELS = {name: MODELS[name] for name in ["AR(2), seasonal diff.", "AR(2), diff. (no seasonal diff.)"]}
SHORT_ORIGINS = ["2025-03-30", "2025-06-29"]

rows = []
for (exog_name, columns), (model_name, model), origin in itertools.product(
        EXOG_SETS.items(), SHORT_MODELS.items(), SHORT_ORIGINS):
    # "none" also starts in January 2024, so it is compared on the same short period as the others
    exog_part = weekly[columns] if columns else None
    try:
        _, info = forecast.forecast_filling(filling_area[weekly.index[0]:], exog=exog_part, train_end=origin,
                                            horizon=26, **model)
    except ValueError as error:
        print(f"Skipped {exog_name}, {model_name}, {origin}: {error}")
        continue
    rows.append({"exog": exog_name, "model": model_name, "origin": origin, "train_weeks": info["train_weeks"],
                 "mae_forecast": info["mae_forecast"], "mae_naive": info["mae_naive"]})

weather_grid = pd.DataFrame(rows)
weather_summary = (weather_grid.groupby(["exog", "model"])
                       .agg(train_weeks=("train_weeks", "mean"), mae_forecast=("mae_forecast", "mean"),
                            mae_naive=("mae_naive", "mean"))
                       .assign(skill=lambda t: 1 - t["mae_forecast"] / t["mae_naive"])
                       .sort_values("mae_forecast"))
weather_summary.round(4)

**Findings from the tests**

*Without weather* (three start points, 26 weeks ahead):
- The best settings were about 6.5 percentage points off on average, against 11.2 for the same week last year: about 40 % better [ ].
- Seasonal differencing (sd = 1) was in every one of the best settings.
- 10 years of history beat both 5 years and all 30. The oldest years seem to add more noise than information.
- A learning rate of 0.01 beat 0.05 in almost every pair.
- The top settings are within 0.2 percentage points of each other. With three start points, that is noise.

*With weather* (2024 to 2025 only, two start points in 2025):
- Precipitation helped the most (about 11 % better than the baseline), temperature a little [ ].
- Without weather, the model was almost exactly the baseline: after the 52-week warm-up there are only about 20 weeks left to learn from.
- Without seasonal differencing, everything was worse.
- The two grids use different start points, so they can't be compared directly.

## The default setting in the app
The app's Forecast page starts on **AR(2) with seasonal differencing (p = 2, sd = 1, m = 52), 10 years of history, learning rate 0.01, 26 weeks ahead, no weather**, with a 90 % interval. AR(2) with MA(1) was marginally better, but the difference was noise, so the simpler model was chosen. Weather is off by default, since it limits the training to 2024 to 2025. Every setting can be changed on the page.

# Weather and reservoirs: sliding window correlation
On the app's *Weather vs reservoirs* page: a weather property against the filling degree or its weekly change, with a lag and window length.

**Findings** (EL 1, weekly change in filling degree, lag 0, 13-week window, 2024 to 2025)

| Weather property | Whole period | Strongest window (+) | Strongest window (−) | Best lag (positive) |
|---|---|---|---|---|
| Temperature | +0.67 | +0.94 | −0.79 | 0 (then a yearly wave) |
| Evaporation | +0.67 | +0.90 | −0.62 | 0 (then a yearly wave) |
| Rain | +0.43 | +0.92 | −0.65 | 0 |
| Precipitation | +0.25 | +0.91 | −0.65 | 0 |
| Snowfall | −0.46 | +0.52 | −0.74 | 12 to 30 weeks, peak at 21 (+0.49) |

- **Every variable changes sign during the year.** Each has windows above +0.9 or +0.5 and below −0.6, so the relation depends strongly on the season.
- **Rain and precipitation** work the same week, most strongly in autumn, and negatively in early summer, when the reservoirs fill from snowmelt instead. Rain correlates better than precipitation, since precipitation also includes snow.
- **Snowfall** is negative the same week (the snow is stored) and positive 12 to 30 weeks later: the snowmelt 3 to 7 months on.
- **Temperature and evaporation** correlate the most, but they mostly measure the season. Their lag scans are an almost perfect yearly wave (+0.67 at lag 0, about −0.7 at half a year, +0.7 at 52 weeks).
- **Over long lags, every variable shows the yearly cycle**, so negative peaks around half a year are the season out of phase, not real effects.

**For predictions:** the seasonal part of SARIMAX already covers the season, so the useful weather is what differs from it: rain at lag 0 and snowfall at about 20 weeks. Temperature and evaporation add little on top of the season. With only two years of weather these are indications, not proof. Removing the season from both series before correlating would show the real effects better, but needs more years.